In [ ]:
import numpy as np
import pandas as pd
import random
import xgboost as xgb
import joblib
from deap import base, creator, tools, algorithms
import time
from collections import defaultdict
from sklearn.preprocessing import StandardScaler

try:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    creator.create("Individual", list, fitness=creator.FitnessMax)
except RuntimeError:
    pass  

random.seed()
np.random.seed()

# 约束定义
def define_constraints():
    return {
        'parameters': {
            'substrate': {'type': 'discrete', 'range': [6, 26, 28, 29]},
            'method': {'type': 'discrete', 'range': [1, 2, 3]},
            'GCI': {'type': 'continuous', 'range': (0, 1)},
            'CPV': {'type': 'continuous', 'range': (-2, 0)},
            'CV+': {'type': 'continuous', 'range': (-0.5, 0.2)},
            'CV-': {'type': 'continuous', 'range': (-2, -0.8)},
            'Ni': {'type': 'continuous', 'range': (0.0001, 2)},
            'Fe': {'type': 'continuous', 'range': (0.0001, 2)},
            'dopant_conc': {'type': 'continuous', 'range': (0, 2)},
            'dopant_Z': {'type': 'discrete', 'range': [0, 15, 16, 24, 25, 27, 42, 50, 74]},
            'Fe_valence': {'type': 'discrete', 'range': [2, 3]},
            'pH': {'type': 'continuous', 'range': (0, 14)},
            'time': {'type': 'continuous', 'range': (10, 3600)}
        },
        'method_dependent_constraints': {
            1: {'required': ['GCI'], 'forbidden': ['CPV', 'CV+', 'CV-']},
            2: {'required': ['CPV'], 'forbidden': ['GCI', 'CV+', 'CV-']},
            3: {'required': ['CV+', 'CV-'], 'forbidden': ['GCI', 'CPV']}
        },
        'fixed_parameters': {
            'c_KOH': 1, 'iR': 0.9, 'cur_density': 10, 'area': 1
        }
    }

constraints = define_constraints()

# 应用约束的函数
def apply_constraints(individual):
    """应用所有约束条件"""
    param_names = list(constraints['parameters'].keys())
    params = dict(zip(param_names, individual))

    for i, name in enumerate(param_names):
        info = constraints['parameters'][name]
        if info['type'] == 'discrete' and individual[i] not in info['range']:
            individual[i] = random.choice(info['range'])

    method = int(round(params['method']))
    if method not in constraints['method_dependent_constraints']:
        method = random.choice(list(constraints['method_dependent_constraints'].keys()))

    method_constraints = constraints['method_dependent_constraints'][method]

    for name in method_constraints['forbidden']:
        idx = param_names.index(name)
        individual[idx] = 0.0

    for name in method_constraints['required']:
        idx = param_names.index(name)
        info = constraints['parameters'][name]
        low, high = info['range']
        if not (low <= individual[idx] <= high) or abs(individual[idx]) < 1e-6:
            if name in ['GCI']:
                individual[idx] = random.uniform(0.01, high)
            elif name in ['CPV', 'CV+', 'CV-']:
                individual[idx] = random.uniform(low, -0.1)

    for elem in ['Ni', 'Fe']:
        idx_elem = param_names.index(elem)
        if individual[idx_elem] <= 0.0001:
            individual[idx_elem] = random.uniform(0.01, 0.1)

    idx_dopant = param_names.index('dopant_conc')
    if params['dopant_Z'] == 0:
        individual[idx_dopant] = 0
    else:
        min_conc = min(individual[param_names.index('Ni')], individual[param_names.index('Fe')])
        if individual[idx_dopant] <= 0:
            individual[idx_dopant] = min_conc * 0.5
        individual[idx_dopant] = min(individual[idx_dopant], min_conc * 1.2)

    idx_valence = param_names.index('Fe_valence')
    if individual[idx_valence] not in constraints['parameters']['Fe_valence']['range']:
        individual[idx_valence] = random.choice(constraints['parameters']['Fe_valence']['range'])

    for i, name in enumerate(param_names):
        info = constraints['parameters'][name]
        if info['type'] == 'continuous':
            low, high = info['range']
            individual[i] = max(low, min(high, individual[i]))

    if method == 3:
        idx_plus = param_names.index('CV+')
        idx_minus = param_names.index('CV-')
        if abs(individual[idx_plus] - individual[idx_minus]) < 0.5:
            individual[idx_minus] = individual[idx_plus] - 0.5
            cvp = constraints['parameters']['CV+']['range']
            cvm = constraints['parameters']['CV-']['range']
            individual[idx_plus] = max(cvp[0], min(cvp[1], individual[idx_plus]))
            individual[idx_minus] = max(cvm[0], min(cvm[1], individual[idx_minus]))

    idx_gci = param_names.index('GCI')
    idx_cpv = param_names.index('CPV')
    idx_cv_plus = param_names.index('CV+')
    idx_cv_minus = param_names.index('CV-')

    if method == 1:
        individual[idx_cpv] = 0.0
        individual[idx_cv_plus] = 0.0
        individual[idx_cv_minus] = 0.0
    elif method == 2:
        individual[idx_gci] = 0.0
        individual[idx_cv_plus] = 0.0
        individual[idx_cv_minus] = 0.0
    elif method == 3:
        individual[idx_gci] = 0.0
        individual[idx_cpv] = 0.0

    return individual


In [2]:
# 创建参数偏好权重函数
def create_preference_functions():
    
    def get_nickel_preference(ni_conc):
        """镍浓度偏好：0.01-0.1为最佳范围"""
        if 0.01 <= ni_conc <= 0.1:
            return 1.3
        elif 0.0001 <= ni_conc < 0.01 :
            return 1.0
        else:
            return 0.7
    
    def get_iron_preference(fe_conc):
        """铁浓度偏好：0.01-0.1为最佳范围"""
        if 0.01 <= fe_conc <= 0.05:
            return 1.2
        elif 0.0001 <= fe_conc < 0.01 :
            return 1
        else:
            return 0.7
    
    def get_gci_preference(gci):
        """恒电流偏好：0.01-0.2为最佳范围"""
        if 0.01 <= gci <= 0.2:
            return 1.2
        elif 0 <= gci < 0.01 or 0.2 < gci <= 1:
            return 1.0
        else:
            return 0.5
    
    def get_voltage_preference(voltage):
        """电压偏好：-1.3到-0.8为最佳范围"""
        if -1.3 <= voltage <= -0.8:
            return 1.2
        elif -2 <= voltage < -1.3:
            return 1.0
        else:
            return 0.7
    
    def get_ph_preference(ph):
        """pH偏好：2-5为最佳范围"""
        if 2 <= ph <= 5:
            return 1.2
        elif 5 < ph <= 6:
            return 1.0
        elif 6 < ph <= 8:
            return 0.8
        else:
            return 0.6
    
    def get_time_preference(time_val):
        """时间偏好：60-900秒为最佳范围"""
        if 60 <= time_val <= 900:
            return 1.2
        elif 10 <= time_val < 60 or 900 < time_val <= 1800:
            return 1.0
        else:
            return 0.8
    
    def get_dopant_preference(dopant_Z, dopant_conc):
        """掺杂偏好：有适量掺杂为佳"""
        if dopant_Z == 0:  # 无掺杂
            return 0.9
        elif 0 < dopant_conc <= 0.1:  # 适量掺杂
            return 1.0   
        else:  # 过量掺杂
            return 0.7
    
    return {
        'Ni': get_nickel_preference,
        'Fe': get_iron_preference,
        'GCI': get_gci_preference,
        'CPV': get_voltage_preference,
        'CV+': get_voltage_preference,
        'CV-': get_voltage_preference,
        'pH': get_ph_preference,
        'time': get_time_preference,
        'dopant': get_dopant_preference
    }

preference_functions = create_preference_functions()


def calculate_preference_weight(individual):
    """计算基于参数偏好的加权平均偏好分数"""
    param_names = list(constraints['parameters'].keys())
    params = dict(zip(param_names, individual))
    
    weights = []  
    
    # 处理普通参数偏好
    for feature, func in preference_functions.items():
        if feature in params and feature != 'dopant':
            weight = func(params[feature])
            weights.append(weight)
    
    # 特殊处理掺杂偏好
    dopant_func = preference_functions['dopant']
    dopant_weight = dopant_func(params['dopant_Z'], params['dopant_conc'])
    weights.append(dopant_weight)
    
    # 计算几何平均作为总权重
    total_weight = np.prod(weights) ** (1 / len(weights))
    
    return total_weight

# 模型加载和预测函数
def load_model_and_scaler(model_path):
    """加载XGBoost模型和标准化器"""

    model = joblib.load(model_path)

    scaler_path = model_path.replace("xgb_model.pkl", "standard_scaler.pkl")
    scaler = joblib.load(scaler_path)
        
    return model, scaler


def prepare_model_input(individual, scaler):
    """将个体转换为模型输入格式"""
    param_names = list(constraints['parameters'].keys())
    params = dict(zip(param_names, individual))
    fixed = constraints['fixed_parameters']
    
    # 按照模型训练时的特征顺序准备输入
    input_features = [
        params['substrate'],
        params['method'], 
        params['GCI'],
        params['CPV'],
        params['CV+'],
        params['CV-'],
        fixed['area'],
        params['time'],
        params['Fe_valence'],
        params['Ni'],
        params['Fe'],
        params['dopant_Z'],
        params['dopant_conc'],
        params['pH'],
        fixed['c_KOH'],
        fixed['iR'],
        fixed['cur_density']
    ]
    
    input_array = np.array(input_features).reshape(1, -1)
    
    # 使用标准化器进行标准化
    input_scaled = scaler.transform(input_array)
    return input_scaled

def predict_performance(model, input_features_scaled):
    """使用模型预测过电位"""
    try:
        if hasattr(model, 'predict'):
            
            overpotential = model.predict(input_features_scaled)[0]
        else:
            
            dmatrix = xgb.DMatrix(input_features_scaled)
            overpotential = model.predict(dmatrix)[0]
        return overpotential
    except Exception as e:
        return 1000  # 返回一个较大的默认值

In [ ]:
_param_names = list(constraints['parameters'].keys())

def _to_param_dict(ind):
    """支持 list 或 dict 的个体转换成 dict"""
    if isinstance(ind, dict):
        return ind.copy()
    return dict(zip(_param_names, ind))

def calculate_distance(ind1, ind2):
    """归一化混合距离（连续变量归一化，离散变量 0/1）"""
    d1 = _to_param_dict(ind1)
    d2 = _to_param_dict(ind2)
    dist_sq = 0.0
    for key, param_info in constraints['parameters'].items():
        v1 = d1[key]
        v2 = d2[key]
        if param_info['type'] == 'continuous':
            low, high = param_info['range']
            norm_diff = 0 if high==low else abs(v1 - v2)/(high-low)
            norm_diff = min(max(norm_diff,0),1)
            dist_sq += norm_diff**2
        else:
            dist_sq += 0 if v1==v2 else 1
    return np.sqrt(dist_sq)

def calculate_max_distance():
    """理论最大距离，用于多样性归一化"""
    n_dims = len(constraints['parameters'])
    return np.sqrt(n_dims)

def calculate_diversity(individual, explored_space, k=5):
    """计算多样性分数"""
    if not explored_space:
        return 1.0
    recent = explored_space[-100:]
    distances = [calculate_distance(individual, e) for e in recent]
    k_use = min(k, len(distances))
    avg_min_distance = np.mean(sorted(distances)[:k_use])
    diversity_score = np.clip(avg_min_distance / calculate_max_distance(), 0, 1)
    return diversity_score

def evaluate_fitness(individual, model, scaler, explored_space=None, 
                   alpha=0.3, mode='development'):
    """评估个体适应度
    
    参数:
    - individual: 待评估的个体
    - model: 预测模型
    - scaler: 标准化器
    - explored_space: 已探索空间（用于多样性计算）
    - alpha: 探索模式中多样性的权重
    - mode: 'development'或'exploration'
    
    返回:
    - 适应度值（元组，DEAP要求）
    """
  
    individual = apply_constraints(individual)
    
  
    input_features_scaled = prepare_model_input(individual, scaler)
    overpotential = predict_performance(model, input_features_scaled)
    
    base_performance = 1 / (1 + np.exp((overpotential - 500) / 100))
    
  
    preference_weight = calculate_preference_weight(individual)
    
    if mode == 'development':
       
        fitness = base_performance * preference_weight
        
    elif mode == 'exploration':
       
        if explored_space is not None:
            diversity_score = calculate_diversity(individual, explored_space)
        else:
            diversity_score = 0.5  
            

        fitness = (alpha * diversity_score + 
                  (1 - alpha) * base_performance * preference_weight)
    
    else:
        raise ValueError(f"未知模式: {mode}")
    
    return (fitness,)  

def adaptive_mutation_rate(generation, max_generations):
    base_rate = 0.1
    decay_factor = np.exp(-3 * generation / max_generations)
    return 0.05 + 0.1 * decay_factor 

In [ ]:
class ElectroDepositionGA:
    def __init__(self, model, scaler, mode='exploration', population_size=30, generations=5):
        """
        电沉积优化GA系统
        
        参数:
        - model: 预测模型
        - scaler: 标准化器
        - mode: 'exploration' 或 'development'
        - population_size: 种群大小
        - generations: 进化代数
        """
        self.model = model
        self.scaler = scaler
        self.mode = mode
        self.population_size = population_size
        self.generations = generations
        
        self.explored_space = [] if mode == 'exploration' else None
        
        self.setup_toolbox()

        self.population = self.toolbox.population(n=population_size)
        self.evaluate_population(self.population)

        self.performance_history = []
        self.best_individuals_history = []
        
        print(f"初始化{mode}模式GA，种群大小: {population_size}，进化代数: {generations}")
    
    def setup_toolbox(self):
        """设置GA工具箱"""
        self.toolbox = base.Toolbox()
        random.seed()
        param_names = list(constraints['parameters'].keys())

        for param_name in param_names:
            param_info = constraints['parameters'][param_name]
            if param_info['type'] == 'continuous':
                low, high = param_info['range']
                self.toolbox.register(f"attr_{param_name}", random.uniform, low, high)
            else:
                self.toolbox.register(f"attr_{param_name}", random.choice, param_info['range'])

        attributes = [getattr(self.toolbox, f"attr_{name}") for name in param_names]
        self.toolbox.register("individual", tools.initCycle, creator.Individual, attributes, n=1)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        if self.mode == 'exploration':
        
            self.toolbox.register("select", tools.selTournament, tournsize=2)
            self.toolbox.register("mate", tools.cxTwoPoint)
            self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.15, indpb=0.4)
        else:
          
            self.toolbox.register("select", tools.selTournament, tournsize=5)
            self.toolbox.register("mate", tools.cxBlend, alpha=0.1)
            self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=0.08, indpb=0.2)
 
        def fitness_function(individual):
            return evaluate_fitness(
                individual, 
                self.model, 
                self.scaler, 
                self.explored_space,
                alpha=0.3 if self.mode == 'exploration' else 0.0,
                mode=self.mode
            )
        
        self.toolbox.register("evaluate", fitness_function)
    
    def evaluate_population(self, population):
        """评估种群"""
        invalid_ind = [ind for ind in population if not ind.fitness.valid]
        fitnesses = self.toolbox.map(self.toolbox.evaluate, invalid_ind)
        for ind, fit in zip(invalid_ind, fitnesses):
            ind.fitness.values = fit
            
            if self.mode == 'exploration':
                self.explored_space.append(ind[:])
    
    def run_optimization(self):
        """运行优化过程"""
        print(f"开始{self.mode}优化...")
        start_time = time.time()
        
        for gen in range(self.generations):

            mutation_rate = adaptive_mutation_rate(gen, self.generations)

            self.run_generation(gen, mutation_rate)

            self.record_performance(gen)

            if gen % 10 == 0 or gen == self.generations - 1:
                self.print_progress(gen)
        
        end_time = time.time()
        print(f"{self.mode}优化完成! 耗时: {end_time - start_time:.2f}秒")
        
        return self.get_results()
    
    def run_generation(self, generation, mutation_rate):
        """运行一代进化"""

        offspring = self.toolbox.select(self.population, len(self.population))
        offspring = list(map(self.toolbox.clone, offspring))

        crossover_rate = 0.8 if self.mode == 'development' else 0.7
        for child1, child2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < crossover_rate:
                self.toolbox.mate(child1, child2)
                del child1.fitness.values
                del child2.fitness.values

        for mutant in offspring:
            if random.random() < mutation_rate:
                self.toolbox.mutate(mutant)
                mutant = apply_constraints(mutant)  # 应用约束
                del mutant.fitness.values

        self.evaluate_population(offspring)
        self.population[:] = offspring
    
    def record_performance(self, generation):
        """记录性能历史"""
        best_individual = tools.selBest(self.population, 1)[0]

        input_features_scaled = prepare_model_input(best_individual, self.scaler)
        overpotential = predict_performance(self.model, input_features_scaled)
        
        performance_score = 1 / (1 + np.exp((overpotential - 500) / 100))
        
        self.performance_history.append({
            'generation': generation,
            'best_fitness': best_individual.fitness.values[0],
            'best_overpotential': overpotential,
            'performance_score': performance_score
        })
        
        self.best_individuals_history.append(best_individual[:])
    
    def print_progress(self, generation):
        """输出优化进度"""
        current_perf = self.performance_history[-1]
        print(f"第{generation}代 - "
              f"适应度: {current_perf['best_fitness']:.4f}, "
              f"过电位: {current_perf['best_overpotential']:.1f}mV")
    
    def get_top_candidates(self, count=10):
        """获取前N个候选个体"""
        sorted_population = sorted(self.population, 
                                 key=lambda ind: ind.fitness.values[0], 
                                 reverse=True)
        
        candidates = []
        for i, individual in enumerate(sorted_population[:count]):
            param_names = list(constraints['parameters'].keys())
            params = dict(zip(param_names, individual))

            input_features_scaled = prepare_model_input(individual, self.scaler)
            overpotential = predict_performance(self.model, input_features_scaled)
            performance_score = 1 / (1 + np.exp((overpotential - 500) / 100))
            preference_weight = calculate_preference_weight(individual)
            diversity = calculate_diversity(individual, self.explored_space) if self.mode == 'exploration' else 0
            
            candidate = {
                'rank': i + 1,
                'parameters': params,
                'fitness': individual.fitness.values[0],
                'overpotential': overpotential,
                'performance_score': performance_score,
                'preference_weight': preference_weight,
                'diversity': diversity,
                'individual': individual[:] 
            }
            candidates.append(candidate)
        
        return candidates
    
    def get_results(self):
        """获取优化结果"""
        top_candidates = self.get_top_candidates(5)                                                 
        
        return {
            'mode': self.mode,
            'top_candidates': top_candidates,
            'performance_history': self.performance_history,
            'best_individual': self.best_individuals_history[-1] if self.best_individuals_history else None,
            'final_population': self.population,
            'explored_space_size': len(self.explored_space) if self.mode == 'exploration' else 0
        }
    
    def export_top_candidates(self, filename="top_candidates.xlsx", count=5):
        """
        导出当前种群中性能最优的候选样本到 Excel 文件。
        适用于 exploration（探索）或 development（开发）阶段。
        """
        import pandas as pd

        if not hasattr(self, "population") or len(self.population) == 0:
            print(" 当前没有可导出的种群，请先运行 run_optimization()。")
            return

        candidates = self.get_top_candidates(count=count)

        rows = []
        for c in candidates:
            row = c["parameters"].copy()
            row["fitness"] = c["fitness"]
            row["overpotential"] = c["overpotential"]
            row["performance_score"] = c["performance_score"]
            row["diversity"] = c["diversity"]
            rows.append(row)

        df = pd.DataFrame(rows)
        df.to_excel(filename, index=False)
        print(f"已将前 {count} 个候选样本保存到 {filename}")

def run_electrodeposition_optimization(model_path, mode='exploration', 
                                     population_size=60, generations=10):
    """
    运行电沉积优化
    
    参数:
    - model_path: 模型路径
    - mode: 'exploration' 或 'development'
    - population_size: 种群大小
    - generations: 进化代数
    """

    model, scaler = load_model_and_scaler(model_path)
    if model is None:
        raise ValueError("模型加载失败")

    ga_system = ElectroDepositionGA(
        model=model,
        scaler=scaler,
        mode=mode,
        population_size=population_size,
        generations=generations
    )
    
    results = ga_system.run_optimization()
    return results, ga_system

def analyze_optimization_results(results):
    """分析优化结果"""
    print(f"\n=== {results['mode']}模式优化结果 ===")
    print(f"探索空间大小: {results.get('explored_space_size', 'N/A')}")
    
    top_candidates = results['top_candidates']
    print(f"\n前5个候选配方:")
    
    for i, candidate in enumerate(top_candidates[:5]):
        print(f"\n#{candidate['rank']} - 适应度: {candidate['fitness']:.4f}, "
              f"过电位: {candidate['overpotential']:.1f}mV")
        
        params = candidate['parameters']
        print("  参数:")
        for param, value in params.items():
            if param in ['substrate', 'method', 'dopant_Z', 'Fe_valence']:
                print(f"    {param}: {value}")
            else:
                print(f"    {param}: {value:.4f}")

    if results['performance_history']:
        initial = results['performance_history'][0]
        final = results['performance_history'][-1]
        improvement = final['best_overpotential'] - initial['best_overpotential']
        
        print(f"\n性能改进:")
        print(f"  初始过电位: {initial['best_overpotential']:.1f}mV")
        print(f"  最终过电位: {final['best_overpotential']:.1f}mV")
        print(f"  改进: {improvement:.1f}mV ({improvement/initial['best_overpotential']*100:.1f}%)")
    
    return top_candidates


